# 04 · 假设检验 Hypothesis Testing

> **能力标签**：SH6（概率与统计 / Probability & statistics）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**原假设**（null hypothesis $H_0$）与**备择假设**（alternative hypothesis $H_1$）的框架。
2. 实现并解释**双样本 t 检验**（two-sample Student's t-test）。
3. 正确定义与解读 **p 值**（p-value）。
4. 区分**第一类错误**（Type I Error, false positive）与**第二类错误**（Type II Error, false negative）。

> 对应能力：**SH6**


## 概念讲解 Concepts

### 假设检验框架 Hypothesis Testing Framework

1. **原假设** $H_0$：默认成立的陈述（如"两组均值相等"）
2. **备择假设** $H_1$：研究者想验证的陈述（如"两组均值不等"）
3. **检验统计量**：将数据压缩为一个数值，度量数据与 $H_0$ 的偏离程度
4. **p 值**：在 $H_0$ 为真的前提下，观测到当前或更极端结果的概率
5. **显著性水平** $\alpha$：通常取 0.05；若 $p < \alpha$ 则拒绝 $H_0$

---

### p 值的正确定义

> **p 值** = $P(\text{观测到当前或更极端的检验统计量} \mid H_0 \text{ 为真})$

**常见误解**：
- ✗ "p 值是 $H_0$ 为真的概率" — **错误**
- ✓ p 值是在 $H_0$ 下出现这么极端数据的概率

---

### 双样本 t 检验 Two-Sample t-test (Student)

假设两独立样本 $a \sim \mathcal{N}(\mu_a, \sigma^2)$，$b \sim \mathcal{N}(\mu_b, \sigma^2)$（等方差假设），$H_0: \mu_a = \mu_b$。

**合并方差**（pooled variance）：
$$s_p^2 = \frac{(n_a-1)s_a^2 + (n_b-1)s_b^2}{n_a+n_b-2}$$

**t 统计量**：
$$t = \frac{\bar{a} - \bar{b}}{s_p\sqrt{\frac{1}{n_a}+\frac{1}{n_b}}}$$

自由度 $df = n_a + n_b - 2$，双侧 $p = 2 \cdot P(T > |t|)$。

---

### 第一/二类错误

|  | 真实 $H_0$ 为真 | 真实 $H_1$ 为真 |
|--|--|--|
| **拒绝 $H_0$** | **第一类错误** (α) | 正确（功效 Power = 1-β）|
| **不拒绝 $H_0$** | 正确 | **第二类错误** (β) |

- $\alpha$（显著性水平）= 第一类错误概率（false positive rate）
- $\beta$ = 第二类错误概率（false negative rate）
- **功效**（Power）= $1 - \beta$


## 示例 Worked Example

In [ ]:
import numpy as np
from scipy import stats

def two_sample_t(a, b) -> dict:
    """双样本等方差 t 检验（Student），手动实现，返回 t 与双侧 p。"""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na, nb = len(a), len(b)
    sp2 = ((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2)
    t = (a.mean() - b.mean()) / np.sqrt(sp2 * (1/na + 1/nb))
    dof = na + nb - 2
    p = 2 * stats.t.sf(abs(t), dof)
    return {"t": float(t), "p": float(p)}

# 示例 1：有真实差异（应该显著）
rng = np.random.default_rng(42)
group_a = rng.normal(0.0, 1.0, 50)
group_b = rng.normal(0.5, 1.0, 50)

result = two_sample_t(group_a, group_b)
print("=== 有差异的两组 ===")
print(f"  均值 A={group_a.mean():.4f}, 均值 B={group_b.mean():.4f}")
print(f"  t = {result['t']:.4f}, p = {result['p']:.4f}")
print(f"  结论: {'拒绝 H₀（显著差异）' if result['p'] < 0.05 else '不拒绝 H₀'} (α=0.05)")

# 对照 scipy 验证
sp_t, sp_p = stats.ttest_ind(group_a, group_b, equal_var=True)
print(f"\n  scipy 对照: t={sp_t:.4f}, p={sp_p:.4f}")
print(f"  与手动实现一致: {np.isclose(result['t'], sp_t) and np.isclose(result['p'], sp_p)}")

# 示例 2：无真实差异（应该不显著）
group_c = rng.normal(0.0, 1.0, 50)
group_d = rng.normal(0.0, 1.0, 50)
result2 = two_sample_t(group_c, group_d)
print(f"\n=== 无差异的两组 ===")
print(f"  t = {result2['t']:.4f}, p = {result2['p']:.4f}")
print(f"  结论: {'拒绝 H₀（显著差异）' if result2['p'] < 0.05 else '不拒绝 H₀（无显著差异）'}")


In [ ]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

# 模拟第一类错误率（α = 0.05 时）
rng = np.random.default_rng(99)
n_sim = 5000
rejections = 0

for _ in range(n_sim):
    a = rng.normal(0, 1, 30)
    b = rng.normal(0, 1, 30)  # H₀ 为真（两组相同）
    _, p = stats.ttest_ind(a, b, equal_var=True)
    if p < 0.05:
        rejections += 1

type1_rate = rejections / n_sim
print(f"H₀ 为真时的拒绝率 ≈ {type1_rate:.4f}")
print(f"理论第一类错误率 α = 0.05")
print(f"（两者应接近）")

# 可视化 p 值分布（H₀ 为真时应为均匀分布）
pvals = []
for _ in range(2000):
    a = rng.normal(0, 1, 30)
    b = rng.normal(0, 1, 30)
    _, p = stats.ttest_ind(a, b, equal_var=True)
    pvals.append(p)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(pvals, bins=20, density=True, color="steelblue", edgecolor="white", alpha=0.8)
ax.axhline(1.0, color="red", ls="--", label="均匀分布（理论）")
ax.axvline(0.05, color="darkorange", ls="--", label="α=0.05")
ax.set_title("H₀ 为真时 p 值分布（应近似均匀）")
ax.set_xlabel("p 值")
ax.set_ylabel("密度")
ax.legend()
fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "pvalue_distribution.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("\n图像已保存到:", _outpath)


## 动手 Exercise

In [ ]:
import numpy as np
from scipy import stats

# 练习：实现 two_sample_t() 并用于真实场景
def two_sample_t(a, b) -> dict:
    """双样本等方差 t 检验，返回 {'t': ..., 'p': ...}。"""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na, nb = len(a), len(b)
    sp2 = ((na - 1)*a.var(ddof=1) + (nb - 1)*b.var(ddof=1)) / (na + nb - 2)
    t = (a.mean() - b.mean()) / np.sqrt(sp2 * (1/na + 1/nb))
    dof = na + nb - 2
    p = 2 * stats.t.sf(abs(t), dof)
    return {"t": float(t), "p": float(p)}

# 模拟场景：比较两种药物的血压降低效果
rng = np.random.default_rng(123)
drug_A = rng.normal(12.0, 3.0, 40)  # 降压均值12
drug_B = rng.normal(10.5, 3.0, 40)  # 降压均值10.5

res = two_sample_t(drug_A, drug_B)
print(f"药物 A: 均值={drug_A.mean():.2f}, SD={drug_A.std(ddof=1):.2f}, n={len(drug_A)}")
print(f"药物 B: 均值={drug_B.mean():.2f}, SD={drug_B.std(ddof=1):.2f}, n={len(drug_B)}")
print(f"\nt = {res['t']:.4f}, p = {res['p']:.4f}")
print(f"结论: {'两种药物效果存在显著差异 (p<0.05)' if res['p']<0.05 else '未发现显著差异'}")


## 延伸阅读 Further Reading

1. **《All of Statistics》— Wasserman**：第 10 章（假设检验）
2. **StatQuest: p-values**（YouTube）— 强烈推荐，直觉解释
3. **ASA Statement on p-values (2016)**：<https://www.amstat.org/publications/jasa/asa-statement-on-p-values>
4. **Effect size 与功效分析**：Cohen's d、功效曲线


## 关联练习 Related Assignments

本课对应练习包：**`w4-ttest`**（实现 `two_sample_t()` 函数）

```bash
python check.py w4-ttest
```

> 能力标签：**SH6** · threshold ≥ 0.7
